# 02 - MNAR vs. MAR Ablation Study

Does the MNAR path beat naive MICE when data is truly MNAR? We evaluate honestly on ground truth.

In [ ]:
from scripts.build_synthetic_benchmarks import generate_benchmark_battery
from umbra.imputers import MARChainedEquationsImputer, HeckmanSelectionImputer
import pandas as pd
import numpy as np

## 1. Ground Truth Setup
In the MNAR benchmark, high income earners disproportionately hide income, and `shadow_z` serves as the instrument.

In [ ]:
battery = generate_benchmark_battery(n_samples=2500, random_state=42)
bench = battery['MNAR_HIGH']
true_mean = bench.true_params['true_mean']
obs_mean = bench.data_observed['income'].mean()
print(f'True Mean: {true_mean:.3f}')
print(f'Observed Mean (Truncated): {obs_mean:.3f} (Selection Bias: {obs_mean - true_mean:+.3f})')

## 2. Naive MAR MICE Imputation
Standard chained equations falsely assume MAR.

In [ ]:
mar = MARChainedEquationsImputer(imputation_method='pmm', random_state=42)
df_mar = mar.fit_transform(bench.data_observed)
mar_mean = df_mar['income'].mean()
print(f'MAR MICE Imputed Mean: {mar_mean:.3f} (Bias: {mar_mean - true_mean:+.3f})')

## 3. Heckman Selection Imputation
MNAR selection model utilizing the shadow variable exclusion restriction.

In [ ]:
heck = HeckmanSelectionImputer(shadow_cols={'income': 'shadow_z'}, random_state=42)
df_heck = heck.fit_transform(bench.data_observed)
heck_mean = df_heck['income'].mean()
print(f'Heckman Imputed Mean: {heck_mean:.3f} (Bias: {heck_mean - true_mean:+.3f})')

## 4. Honest Summary Table

In [ ]:
summary_df = pd.DataFrame([
    {'Method': 'Observed Naive', 'Mean': obs_mean, 'Bias': obs_mean - true_mean},
    {'Method': 'MAR MICE (PMM)', 'Mean': mar_mean, 'Bias': mar_mean - true_mean},
    {'Method': 'Heckman Selection', 'Mean': heck_mean, 'Bias': heck_mean - true_mean},
])
summary_df